In [47]:
import pandas as pd
from unidecode import unidecode
import geopandas as gpd
import pydeck as pdk
import json
import webbrowser
from pathlib import Path
from datetime import datetime
import os
from functools import reduce
import duckdb
import numpy as np        
import folium
from folium import plugins
hoy = datetime.today()


usuario = os.getlogin()
current_date = datetime.now().strftime("%d-%m-%y")

In [48]:
base_tode = pd.read_parquet(fr"C:\Users\{usuario}\IMSS-BIENESTAR/División de Procesamiento de información - Repositorio de Datos/Productividad/conteos con ece/todes_consolidadas.parquet")
base_metas = pd.read_excel(rf"C:\Users\{usuario}\IMSS-BIENESTAR/División de Procesamiento de información - Repositorio de Datos\Productividad\Metas\2026\Metas de productividad por unidad medica 2026.xlsx")
clues = pd.read_parquet(fr"C:\Users\{usuario}\IMSS-BIENESTAR\División de Procesamiento de información - Repositorio de Datos\CLUES\clues.parquet")


In [49]:
unicos = base_tode['tipo_consulta'].unique()
unicos

array(['general', 'especialidad', 'qx', 'egresos'], dtype=object)

In [50]:
consu= base_tode[base_tode['anio_insert'] == "2026"]

In [51]:
consu = consu[
    consu['tipo_consulta'].isin(['general', 'especialidad', 'qx'])
]

In [52]:
consu['tipo_consulta'] = consu['tipo_consulta'].replace({
    'qx': 'cirugia'
})

In [55]:
cirugia = consu.loc[
    consu['tipo_consulta'] == 'cirugia',
    'procedimientos'
].sum()

consulta = consu.loc[
    consu['tipo_consulta'].isin(['general', 'especialidad']),
    'procedimientos'
].sum()
general = consu.loc[
    consu['tipo_consulta'] == 'general',
    'procedimientos'
].sum()
especialidad = consu.loc[
    consu['tipo_consulta'] == 'especialidad',
    'procedimientos'
].sum()
conteo_consu = pd.DataFrame({
    'cirugia': [cirugia],
    'consulta': [consulta],
    'general': [general],
    'especialidad': [especialidad]
})

In [ ]:
break

In [56]:
conteo_consu

,cirugia,consulta,general,especialidad
0,357518,22791740,20023894,2767846


In [ ]:
query = """
WITH base AS (
    SELECT
        clues AS clues_imb,

        SUM(CASE WHEN tipo_consulta = 'general' THEN procedimientos ELSE 0 END) AS general,
        SUM(CASE WHEN tipo_consulta = 'qx' THEN procedimientos ELSE 0 END) AS qx,
        SUM(CASE WHEN tipo_consulta = 'especialidad' THEN procedimientos ELSE 0 END) AS especialidad,
        SUM(CASE WHEN tipo_consulta = 'egresos' THEN procedimientos ELSE 0 END) AS egresos

    FROM base_tode
    WHERE anio_insert = 2026
    GROUP BY clues
),

metas AS (
    SELECT
        entidad,
        clues_imb,
        estatus_de_operacion,
        nombre_de_la_unidad,
        nivel_atencion,
        meta_general_anual,
        meta_especialidad_anual,
        meta_cirugia_anual,
        meta_egresos_anual
    FROM base_metas
)

SELECT
    m.clues_imb,
    m.entidad,
    m.estatus_de_operacion,
    m.nombre_de_la_unidad,
    m.nivel_atencion,

    b.general,
    b.qx,
    b.especialidad,
    b.egresos,

    m.meta_general_anual,
    m.meta_especialidad_anual,
    m.meta_cirugia_anual,
    m.meta_egresos_anual

FROM metas m
LEFT JOIN base b
    ON m.clues_imb = b.clues_imb
"""
base = duckdb.query(query).to_df()
base = base.rename(columns={
    "qx": "cirugias",})

In [ ]:
cols = [
    'clues_imb', 'entidad', 'estatus_de_operacion', 'nombre_de_la_unidad',
    'nivel_atencion', 'general', 'cirugias', 'especialidad', 'egresos',
    'meta_general_anual', 'meta_especialidad_anual',
    'meta_cirugia_anual', 'meta_egresos_anual'
]

base = base[cols].copy()


In [ ]:
base_metas

,entidad,clues_imb,categoria_gerencial,estatus_de_operacion,nombre_de_la_unidad,nivel_atencion,meta_general_anual,meta_especialidad_anual,meta_cirugia_anual,meta_egresos_anual
0,BAJA CALIFORNIA,BCIMB000010,Generales 100-149 c,EN OPERACION,HOSPITAL GENERAL DE ENSENADA,SEGUNDO NIVEL,0,59157,6095,7305
1,BAJA CALIFORNIA,BCIMB000022,Unidades moviles,EN OPERACION,UNIDAD MÓVIL NO. 7,PRIMER NIVEL,1899,0,0,0
2,BAJA CALIFORNIA,BCIMB000034,Unidades moviles,EN OPERACION,UNIDAD MÓVIL NO. 8,PRIMER NIVEL,1899,0,0,0
3,BAJA CALIFORNIA,BCIMB000046,Unidades moviles,EN OPERACION,UNIDAD MÓVIL 9,PRIMER NIVEL,1899,0,0,0
4,BAJA CALIFORNIA,BCIMB000051,1-2 nucleos,EN OPERACION,COLONIA LOMA LINDA,PRIMER NIVEL,3708,0,0,0
...,...,...,...,...,...,...,...,...,...,...
10573,ZACATECAS,ZSIMB002621,Unidades moviles,EN OPERACION,CARAVANA DE LA SALUD TIPO 0 CIENEGUILLA (NORIA...,NO APLICA,1694,0,0,0
10574,ZACATECAS,ZSIMB002633,Unidades moviles,EN OPERACION,CARAVANA DE LA SALUD TIPO 0 LA VILLITA,NO APLICA,1694,0,0,0
10575,ZACATECAS,ZSIMB002650,6-12 nucleos,EN OPERACION,CENTRO DE SALUD JEREZ,PRIMER NIVEL,12428,0,0,0
10576,CHIAPAS,CSIMB003902,Servicios ampliados,EN OPERACION,CLINICA PARA LA ATENCION DE PARTO HUMANIZADO S...,SEGUNDO NIVEL,0,2301,0,0


In [ ]:
conteo_consultas = (
    base
    .agg({
        'cirugias': 'sum',
        'general': 'sum',
        'especialidad': 'sum'
    })
)

conteo_consultas = pd.DataFrame({
    'cirugia': [conteo_consultas['cirugias']],
    'consulta': [conteo_consultas['general'] + conteo_consultas['especialidad']]
})

In [ ]:
conteo_consultas

,cirugia,consulta
0,356106.0,22725087.0


In [ ]:

#  consultas según nivel de atención
base['consultas'] = np.where(
    base['nivel_atencion'].isin(['SEGUNDO NIVEL', 'TERCER NIVEL']),
    base['cirugias'],
    base[['general','especialidad']].sum(axis=1)
)

#  meta total (siempre suma de todas las metas)
base['meta_total'] = (
    base['meta_general_anual']
    + base['meta_especialidad_anual']
    + base['meta_cirugia_anual']
    + base['meta_egresos_anual']
)

In [ ]:
cols_base =['clues_imb', 'entidad', 'nombre_de_la_unidad',
       'nivel_atencion', 'cirugias','meta_cirugia_anual','consultas', 'meta_total']
base = base[cols_base].copy()   

In [ ]:
base = base.merge(
    clues[['clues_imb', 'latitud', 'longitud']], 
    on='clues_imb',
    how='left'
)

In [ ]:
columnas_finales = [
 'clues_imb', 'entidad', 'nombre_de_la_unidad', 'nivel_atencion',
       'cirugias', 'meta_cirugia_anual', 'consultas', 'meta_total', 'latitud',
       'longitud'
]
base = base.drop_duplicates(subset=columnas_finales, keep="first")

In [ ]:
from datetime import timedelta

dias_desde_miercoles = (hoy.weekday() - 2) % 7
if dias_desde_miercoles == 0:
    dias_desde_miercoles = 7
else:
    dias_desde_miercoles += 7
ultimo_miercoles = hoy - timedelta(days=dias_desde_miercoles)

dias_transcurridos = ultimo_miercoles.timetuple().tm_yday
dias_del_anio = 365 + int(ultimo_miercoles.year % 4 == 0 and (ultimo_miercoles.year % 100 != 0 or ultimo_miercoles.year % 400 == 0))

base['pct_tiempo'] = dias_transcurridos / dias_del_anio
base['nivel_atencion'] = base['nivel_atencion'].str.strip().str.upper()
mask = base['nivel_atencion'].isin(['SEGUNDO NIVEL', 'TERCER NIVEL'])

base.loc[mask, 'meta_esperada_ciru'] = (
    base.loc[mask, 'meta_cirugia_anual'] * base.loc[mask, 'pct_tiempo']
 )
mask = base['nivel_atencion'].isin(['SEGUNDO NIVEL', 'TERCER NIVEL'])

base['pct_cirugia'] = np.where(
    mask,
    base['cirugias'] / base['meta_esperada_ciru'] * 100,
    np.nan
)

In [ ]:
mask_consulta = ~base['nivel_atencion'].isin(['SEGUNDO NIVEL', 'TERCER NIVEL'])

base.loc[mask_consulta, 'meta_esperada_consulta'] = (
    base.loc[mask_consulta, 'meta_total'] * base.loc[mask_consulta, 'pct_tiempo']
)

In [ ]:
base['pct_consulta'] = np.where(
    mask_consulta,
    base['consultas'] / base['meta_esperada_consulta'] * 100,
    np.nan
)

In [ ]:
import numpy as np

mask_cirugia = base['nivel_atencion'].isin(['SEGUNDO NIVEL', 'TERCER NIVEL'])
mask_consulta = ~mask_cirugia

base['pct_general'] = np.where(
    mask_cirugia,
    base['cirugias'] / base['meta_esperada_ciru'] * 100,
    base['consultas'] / base['meta_esperada_consulta'] * 100
)

In [ ]:
base['pct_general'] = base['pct_general'].round(1)

In [ ]:
base.columns

Index(['clues_imb', 'entidad', 'nombre_de_la_unidad', 'nivel_atencion',
       'cirugias', 'meta_cirugia_anual', 'consultas', 'meta_total', 'latitud',
       'longitud', 'pct_tiempo', 'meta_esperada_ciru', 'pct_cirugia',
       'meta_esperada_consulta', 'pct_consulta', 'pct_general'],
      dtype='object')

In [ ]:
suma_cirugias = base['cirugias'].sum()
suma_consultas = base['consultas'].sum()

print(f'Cirugías: {suma_cirugias}')
print(f'Consultas: {suma_consultas}')

Cirugías: 356106.0
Consultas: 18485844.0


In [ ]:
#break

## mapa

In [ ]:

def asignar_macro_categoria(nivel_atencion):
    if nivel_atencion in ["SEGUNDO NIVEL", "TERCER NIVEL"]:
        return "cirugias"
    else:
        return "consultas"

semaforo_colores = {
    "rojo": "#D41111",
    "amarillo": "#F1D54A",
    "verde claro": "#88A91E",
    "verde fuerte": "#0D5D2A"
}

PCT_ANIO_PORCENTAJE = 0

def obtener_color_semaforo(avance):
    global PCT_ANIO_PORCENTAJE
    
    if pd.isna(avance) or np.isinf(avance):
        return semaforo_colores["rojo"]
    
    if PCT_ANIO_PORCENTAJE == 0:
        if avance >= 95:
            return semaforo_colores["verde fuerte"]
        elif avance >= 80:
            return semaforo_colores["verde claro"]
        elif avance >= 60:
            return semaforo_colores["amarillo"]
        else:
            return semaforo_colores["rojo"]
    
    verde_fuerte_threshold = PCT_ANIO_PORCENTAJE * 0.95
    verde_claro_threshold = PCT_ANIO_PORCENTAJE * 0.80
    amarillo_threshold = PCT_ANIO_PORCENTAJE * 0.60
    
    if avance >= verde_fuerte_threshold:
        return semaforo_colores["verde fuerte"]
    elif avance >= verde_claro_threshold:
        return semaforo_colores["verde claro"]
    elif avance >= amarillo_threshold:
        return semaforo_colores["amarillo"]
    else:
        return semaforo_colores["rojo"]

def obtener_estado_semaforo(avance):
    global PCT_ANIO_PORCENTAJE
    
    if pd.isna(avance) or np.isinf(avance):
        return "Sin datos validos"
    
    verde_fuerte_threshold = PCT_ANIO_PORCENTAJE * 0.95
    verde_claro_threshold = PCT_ANIO_PORCENTAJE * 0.80
    amarillo_threshold = PCT_ANIO_PORCENTAJE * 0.60
    
    if avance >= verde_fuerte_threshold:
        return f"Verde Fuerte (Avance >= {verde_fuerte_threshold:.1f}%)"
    elif avance >= verde_claro_threshold:
        return f"Verde Claro (Avance >= {verde_claro_threshold:.1f}%)"
    elif avance >= amarillo_threshold:
        return f"Amarillo (Avance >= {amarillo_threshold:.1f}%)"
    else:
        return f"Rojo (Avance < {amarillo_threshold:.1f}%)"

In [ ]:

df_plot = base.copy()

df_plot["latitud"] = pd.to_numeric(df_plot["latitud"], errors="coerce")
df_plot["longitud"] = pd.to_numeric(df_plot["longitud"], errors="coerce")
df_plot["pct_general"] = pd.to_numeric(df_plot["pct_general"], errors="coerce")
df_plot["cirugias"] = pd.to_numeric(df_plot["cirugias"], errors="coerce").fillna(0)
df_plot["consultas"] = pd.to_numeric(df_plot["consultas"], errors="coerce").fillna(0)
df_plot["meta_cirugia_anual"] = pd.to_numeric(df_plot["meta_cirugia_anual"], errors="coerce").fillna(1)
df_plot["meta_total"] = pd.to_numeric(df_plot["meta_total"], errors="coerce").fillna(1)
df_plot["meta_esperada_ciru"] = pd.to_numeric(df_plot["meta_esperada_ciru"], errors="coerce").fillna(1)
df_plot["meta_esperada_consulta"] = pd.to_numeric(df_plot["meta_esperada_consulta"], errors="coerce").fillna(1)

df_plot["pct_general"] = df_plot["pct_general"].replace([np.inf, -np.inf], np.nan)

df_plot = df_plot.dropna(subset=["latitud", "longitud", "pct_general"]).copy()
df_plot["pct_general"] = df_plot["pct_general"].astype(float)

In [ ]:
if "pct_tiempo" in df_plot.columns and len(df_plot) > 0:
    pct_tiempo_decimal = pd.to_numeric(df_plot["pct_tiempo"].iloc[0], errors="coerce")
    if pd.isna(pct_tiempo_decimal) or np.isinf(pct_tiempo_decimal):
        pct_tiempo_decimal = 0
else:
    pct_tiempo_decimal = 0

pct_dia_esperado = pct_tiempo_decimal * 100
PCT_ANIO_PORCENTAJE = pct_dia_esperado

In [ ]:


df_plot["pct_dia"] = pct_dia_esperado
df_plot["pct_dia_fmt"] = df_plot["pct_dia"].map(lambda x: f"{x:.1f}%")

df_plot["macro_key"] = df_plot["nivel_atencion"].apply(asignar_macro_categoria)

df_plot["color_hex"] = df_plot["pct_general"].apply(obtener_color_semaforo)

df_plot["pct_general_fmt"] = df_plot["pct_general"].map(
    lambda x: f"{x:.1f}%" if not pd.isna(x) else "0.0%"
)

df_plot["estado_semaforo"] = df_plot["pct_general"].apply(obtener_estado_semaforo)

In [ ]:


def formatear_metricas(row):
    if row["nivel_atencion"] in ["SEGUNDO NIVEL", "TERCER NIVEL"]:
        realizado = row.get("cirugias", 0)
        if pd.isna(realizado) or np.isinf(realizado):
            realizado = 0
        meta_anual = row.get("meta_cirugia_anual", 1)
        if pd.isna(meta_anual) or np.isinf(meta_anual):
            meta_anual = 1
        meta_esperada = row.get("meta_esperada_ciru", 1)
        if pd.isna(meta_esperada) or np.isinf(meta_esperada):
            meta_esperada = 1
        tipo = "cirugias"
    else:
        realizado = row.get("consultas", 0)
        if pd.isna(realizado) or np.isinf(realizado):
            realizado = 0
        meta_anual = row.get("meta_total", 1)
        if pd.isna(meta_anual) or np.isinf(meta_anual):
            meta_anual = 1
        meta_esperada = row.get("meta_esperada_consulta", 1)
        if pd.isna(meta_esperada) or np.isinf(meta_esperada):
            meta_esperada = 1
        tipo = "consultas"
    
    return pd.Series({
        "realizado_fmt": f"{int(realizado):,}",
        "meta_anual_fmt": f"{int(meta_anual):,}",
        "meta_esperada_fmt": f"{int(meta_esperada):,}",
        "tipo_indicador": tipo
    })

df_plot[
    ["realizado_fmt", "meta_anual_fmt", "meta_esperada_fmt", "tipo_indicador"]
] = df_plot.apply(formatear_metricas, axis=1)


In [ ]:
df_validos = df_plot[df_plot["pct_general"].notna()].copy()
df_validos = df_validos[~np.isinf(df_validos["pct_general"])].copy()

df_sin_ceros = df_validos[df_validos["pct_general"] > 0].copy()

df_cirugias = df_validos[df_validos["macro_key"] == "cirugias"].copy()
df_consultas = df_validos[df_validos["macro_key"] == "consultas"].copy()

total_unidades_activas = len(df_validos)

unidades_cirugias = len(df_sin_ceros[df_sin_ceros["macro_key"] == "cirugias"])
unidades_consultas = len(df_sin_ceros[df_sin_ceros["macro_key"] == "consultas"])

if len(df_cirugias) > 0:
    total_cirugias = df_cirugias["cirugias"].sum()
    total_cirugias = total_cirugias if not pd.isna(total_cirugias) else 0
    meta_cirugias_total = df_cirugias["meta_cirugia_anual"].sum()
    meta_cirugias_total = meta_cirugias_total if not pd.isna(meta_cirugias_total) else 0
else:
    total_cirugias = 0
    meta_cirugias_total = 0

if len(df_consultas) > 0:
    total_consultas = df_consultas["consultas"].sum()
    total_consultas = total_consultas if not pd.isna(total_consultas) else 0
    meta_consultas_total = df_consultas["meta_total"].sum()
    meta_consultas_total = meta_consultas_total if not pd.isna(meta_consultas_total) else 0
else:
    total_consultas = 0
    meta_consultas_total = 0

total_realizado = total_cirugias + total_consultas
meta_anual_total = meta_cirugias_total + meta_consultas_total

if meta_anual_total > 0:
    avg_avance_general = total_realizado / meta_anual_total * 100
    avg_avance_general = avg_avance_general if not pd.isna(avg_avance_general) else 0
else:
    avg_avance_general = 0

if meta_cirugias_total > 0:
    avg_avance_cirugias = total_cirugias / meta_cirugias_total * 100
    avg_avance_cirugias = avg_avance_cirugias if not pd.isna(avg_avance_cirugias) else 0
else:
    avg_avance_cirugias = 0

if meta_consultas_total > 0:
    avg_avance_consultas = total_consultas / meta_consultas_total * 100
    avg_avance_consultas = avg_avance_consultas if not pd.isna(avg_avance_consultas) else 0
else:
    avg_avance_consultas = 0

In [ ]:

unidades_data = []
for _, row in df_validos.iterrows():
    unidades_data.append({
        "clues": str(row["clues_imb"]),
        "nombre": str(row["nombre_de_la_unidad"]),
        "lat": float(row["latitud"]),
        "lng": float(row["longitud"]),
        "avance": row["pct_general_fmt"],
        "estado": row["estado_semaforo"]
    })

In [ ]:
# # mapa jsjsjsjs
# mapa_center_lat = df_validos["latitud"].mean()
# mapa_center_lon = df_validos["longitud"].mean()

# m = folium.Map(  # puro pinche folium y no mmdas jsjsjsjsjs
#     location=[mapa_center_lat, mapa_center_lon],
#     zoom_start=6,
#     tiles="CartoDB dark_matter",
#     control_scale=True
# )

In [ ]:
break

SyntaxError: 'break' outside loop (668683560.py, line 1)

## html

In [ ]:
break

SyntaxError: 'break' outside loop (668683560.py, line 1)

## correr

In [ ]:
output_path = Path.home() / "Downloads" / "dashboard_productividad_unidades.html"

m.save(str(output_path))

webbrowser.open(str(output_path))

True

## intento de mapa 3d

In [ ]:

# ============================================================
# PREPARACION DE DATOS PARA 3D (USANDO TU SEMÁFORO)
# ============================================================

df_3d = df_validos.copy()

# Calcular el porcentaje del año transcurrido (como ya lo haces)
if "pct_tiempo" in df_3d.columns and len(df_3d) > 0:
    pct_tiempo_decimal = pd.to_numeric(df_3d["pct_tiempo"].iloc[0], errors="coerce")
    if pd.isna(pct_tiempo_decimal) or np.isinf(pct_tiempo_decimal):
        pct_tiempo_decimal = 0
else:
    pct_tiempo_decimal = 0

pct_dia_esperado = pct_tiempo_decimal * 100  # Esto da 41.9% en tu ejemplo

# ¡IMPORTANTE! Actualizar la variable global para tus funciones
PCT_ANIO_PORCENTAJE = pct_dia_esperado

# Ahora sí, procesar los datos numéricos
if df_3d["pct_general"].dtype == "object":
    df_3d["pct_general_num"] = pd.to_numeric(df_3d["pct_general"].str.rstrip('%'), errors='coerce') / 100
else:
    if df_3d["pct_general"].max() > 1:
        df_3d["pct_general_num"] = df_3d["pct_general"] / 100
    else:
        df_3d["pct_general_num"] = df_3d["pct_general"]

df_3d["pct_general_num"] = df_3d["pct_general_num"].fillna(0).clip(0, 1)

# Altura de las columnas
df_3d["altura"] = 500 + (df_3d["pct_general_num"] * 20000)

# ========== USAR TU FUNCIÓN DE SEMÁFORO ==========
def hex_to_rgb(hex_color):
    hex_color = hex_color.lstrip('#')
    return tuple(int(hex_color[i:i+2], 16) for i in (0, 2, 4))

# Aplicar tu función obtener_color_semaforo (que ya usa PCT_ANIO_PORCENTAJE)
df_3d["color_semaforo_hex"] = df_3d["pct_general"].apply(obtener_color_semaforo)

# Convertir a RGB para deck.gl
df_3d[["r", "g", "b"]] = df_3d["color_semaforo_hex"].apply(
    lambda x: pd.Series(hex_to_rgb(x))
)

# Calcular los umbrales para mostrar en el HTML (los mismos que usa tu semáforo)
verde_fuerte_umbral = PCT_ANIO_PORCENTAJE * 0.95
verde_claro_umbral = PCT_ANIO_PORCENTAJE * 0.80
amarillo_umbral = PCT_ANIO_PORCENTAJE * 0.60

volumen_cirugias = pd.to_numeric(conteo_consultas.loc[0, "cirugia"], errors="coerce")
volumen_consultas = pd.to_numeric(conteo_consultas.loc[0, "consulta"], errors="coerce")

volumen_cirugias = 0 if pd.isna(volumen_cirugias) else volumen_cirugias
volumen_consultas = 0 if pd.isna(volumen_consultas) else volumen_consultas
volumen_realizado = volumen_cirugias + volumen_consultas

# Datos para el buscador (incluyendo el color correcto)
unidades_data = []
for _, row in df_3d.iterrows():
    unidades_data.append({
        "clues": row['clues_imb'],
        "nombre": row['nombre_de_la_unidad'],
        "lat": row['latitud'],
        "lng": row['longitud'],
        "avance": row['pct_general_fmt'],
        "entidad": row['entidad'],
        "nivel": row['nivel_atencion'],
        "tipo": row['tipo_indicador'],
        "realizado": row['realizado_fmt'],
        "meta_anual": row['meta_anual_fmt'],
        "meta_esperada": row['meta_esperada_fmt'],
        "pct_dia_fmt": row['pct_dia_fmt'],
        "color_hex": row['color_semaforo_hex'],  # ← Color del semáforo
        "estado_semaforo": row['estado_semaforo'],
        "nombre_de_la_unidad": row['nombre_de_la_unidad'],
        "altura": row['altura'],
        "r": row['r'],
        "g": row['g'],
        "b": row['b']
    })
unidades_data_json = json.dumps(unidades_data, ensure_ascii=False)
# Calcular centro del mapa
lat_centro = df_3d["latitud"].mean()
lon_centro = df_3d["longitud"].mean()
# ============================================================
# GENERAR HTML 3D - VERSIÓN CON COLORES FIJOS DEL SEMÁFORO
# ============================================================

html_3d = f"""
<!DOCTYPE html>
<html>
<head>
<meta charset="utf-8"/>
<title>Hospital Dashboard 3D - Productividad por Unidad Medica</title>

<script src="https://unpkg.com/deck.gl@latest/dist.min.js"></script>
<script src="https://unpkg.com/maplibre-gl@latest/dist/maplibre-gl.js"></script>
<link href="https://unpkg.com/maplibre-gl@latest/dist/maplibre-gl.css" rel="stylesheet"/>

<style>
html, body {{
    margin: 0;
    width: 100%;
    height: 100%;
    font-family: 'Segoe UI', Arial, sans-serif;
    background: #0a0a0a;
}}

#map {{
    width: 100%;
    height: 100%;
    background: #0d0d0d;
}}

.header {{
    position: fixed;
    top: 0;
    left: 0;
    right: 0;
    z-index: 10000;
    background: linear-gradient(135deg, #0a0e27, #0f142e);
    color: white;
    padding: 12px 25px;
    font-size: 18px;
    font-weight: bold;
    text-align: center;
    box-shadow: 0 2px 10px rgba(0,0,0,0.5);
    pointer-events: none;
    border-bottom: 1px solid rgba(78,205,196,0.3);
}}

.kpi-container {{
    position: fixed;
    top: 55px;
    left: 0;
    right: 0;
    z-index: 10000;
    display: grid;
    grid-template-columns: repeat(4, 1fr);
    gap: 12px;
    padding: 12px 20px;
    pointer-events: none;
}}

.kpi-card {{
    background: rgba(10,10,10,0.85);
    backdrop-filter: blur(10px);
    padding: 10px 15px;
    border-radius: 10px;
    border: 1px solid rgba(78,205,196,0.3);
    pointer-events: auto;
    box-shadow: 0 2px 5px rgba(0,0,0,0.3);
}}

.kpi-label {{
    font-size: 10px;
    color: #4ecdc4;
    text-transform: uppercase;
    letter-spacing: 1px;
}}

.kpi-value {{
    font-size: 24px;
    font-weight: bold;
    color: white;
}}

.kpi-sub {{
    font-size: 9px;
    color: #888;
}}

.dashboard-btn {{
    position: fixed;
    top: 350px;
    left: 20px;
    z-index: 10001;
}}

.dashboard-btn button {{
    background: linear-gradient(135deg, #4ecdc4, #2c3e50);
    color: white;
    border: none;
    padding: 8px 18px;
    border-radius: 25px;
    font-size: 12px;
    font-weight: bold;
    cursor: pointer;
    font-family: 'Segoe UI', Arial, sans-serif;
    box-shadow: 0 2px 5px rgba(0,0,0,0.3);
    transition: all 0.3s;
}}

.dashboard-btn button:hover {{
    transform: translateY(-2px);
    box-shadow: 0 4px 10px rgba(78,205,196,0.3);
}}

.buscador {{
    position: fixed;
    top: 200px;
    left: 20px;
    z-index: 10001;
    width: 320px;
}}

.buscador-box {{
    background: rgba(10,10,10,0.85);
    backdrop-filter: blur(10px);
    border-radius: 12px;
    padding: 12px;
    border: 1px solid rgba(78,205,196,0.3);
    box-shadow: 0 4px 15px rgba(0,0,0,0.5);
}}

.buscador-label {{
    color: #4ecdc4;
    font-size: 10px;
    text-transform: uppercase;
    margin-bottom: 8px;
    letter-spacing: 1px;
}}

.buscador-input {{
    width: 100%;
    padding: 10px 15px;
    background: rgba(20,20,20,0.9);
    border: 1px solid rgba(78,205,196,0.3);
    border-radius: 8px;
    color: white;
    font-size: 13px;
    outline: none;
    font-family: 'Segoe UI', Arial, sans-serif;
    box-sizing: border-box;
}}

.buscador-input:focus {{
    border-color: #4ecdc4;
    box-shadow: 0 0 5px rgba(78,205,196,0.5);
}}

.resultados {{
    background: rgba(10,10,10,0.95);
    border-radius: 12px;
    margin-top: 8px;
    max-height: 350px;
    overflow-y: auto;
    display: none;
    border: 1px solid rgba(78,205,196,0.3);
    box-shadow: 0 4px 15px rgba(0,0,0,0.5);
}}

.resultado-item {{
    padding: 12px 15px;
    cursor: pointer;
    border-bottom: 1px solid rgba(78,205,196,0.1);
    font-family: 'Segoe UI', Arial, sans-serif;
    transition: all 0.2s;
}}

.resultado-item:hover {{
    background: rgba(78,205,196,0.15);
    border-left: 3px solid #4ecdc4;
}}

.semaforo {{
    position: fixed;
    bottom: 20px;
    right: 20px;
    z-index: 10001;
    background: rgba(10,10,10,0.85);
    backdrop-filter: blur(10px);
    padding: 12px 18px;
    border-radius: 12px;
    min-width: 240px;
    border: 1px solid rgba(78,205,196,0.3);
    box-shadow: 0 4px 15px rgba(0,0,0,0.5);
    pointer-events: auto;
}}

.footer {{
    position: fixed;
    bottom: 20px;
    left: 20px;
    z-index: 10001;
    background: rgba(10,10,10,0.8);
    backdrop-filter: blur(10px);
    padding: 8px 12px;
    border-radius: 8px;
    font-size: 10px;
    color: #888;
    border: 1px solid rgba(78,205,196,0.2);
    pointer-events: auto;
}}

.deck-tooltip {{
    background: rgba(0,0,0,0.85) !important;
    backdrop-filter: blur(8px) !important;
    border: 1px solid rgba(78,205,196,0.4) !important;
    border-radius: 8px !important;
    box-shadow: 0 4px 15px rgba(0,0,0,0.5) !important;
    padding: 0 !important;
}}

.seleccion-activa {{
    position: fixed;
    top: 420px;
    left: 20px;
    z-index: 10001;
    background: rgba(78,205,196,0.2);
    backdrop-filter: blur(10px);
    padding: 5px 12px;
    border-radius: 20px;
    border: 1px solid #4ecdc4;
    color: #4ecdc4;
    font-size: 11px;
    font-weight: bold;
    pointer-events: none;
}}
</style>
</head>
<body>

<div id="map"></div>

<div class="header">
    Hospital Dashboard - Productividad por Unidad Medica (3D)
</div>

<div class="kpi-container">
    <div class="kpi-card">
        <div class="kpi-label">Unidades Activas</div>
        <div class="kpi-value">{total_unidades_activas:,}</div>
        <div class="kpi-sub">Cirugia: {unidades_cirugias} | Consulta: {unidades_consultas}</div>
    </div>
    <div class="kpi-card">
        <div class="kpi-label">Avance General</div>
        <div class="kpi-value">{avg_avance_general:.1f}%</div>
        <div class="kpi-sub">Cirugia: {avg_avance_cirugias:.1f}% | Consulta: {avg_avance_consultas:.1f}%</div>
    </div>
    <div class="kpi-card">
        <div class="kpi-label">Volumen Realizado</div>
        <div class="kpi-value">{volumen_realizado:,.0f}</div>
        <div class="kpi-sub">Cirugias: {volumen_cirugias:,.0f} | Consultas: {volumen_consultas:,.0f}</div>
    </div>
    <div class="kpi-card">
        <div class="kpi-label">Cobertura</div>
        <div class="kpi-value">{df_validos['entidad'].nunique()}</div>
        <div class="kpi-sub">Entidades Federativas</div>
    </div>
</div>

<div class="dashboard-btn">
    <button onclick="window.open('https://argontc.shinyapps.io/pptx/', '_blank')">
        Ver Dashboard de Reportes
    </button>
</div>

<div class="buscador">
    <div class="buscador-box">
        <div class="buscador-label">Buscador de Unidades</div>
        <input type="text" id="buscadorClues" class="buscador-input" placeholder="Buscar por CLUES o nombre...">
    </div>
    <div id="resultadosBusqueda" class="resultados"></div>
</div>

<div id="seleccionInfo" class="seleccion-activa" style="display: none;"></div>

<div class="semaforo">
    <div style="color: white; font-size: 13px; font-weight: bold; margin-bottom: 10px; text-align: center; border-bottom: 1px solid rgba(78,205,196,0.3); padding-bottom: 5px;">
        Semaforo de Avance
    </div>
    <div style="font-size: 11px; color: #4ecdc4; margin-bottom: 10px; text-align: center; background: rgba(0,0,0,0.5); padding: 5px; border-radius: 6px;">
        Año transcurrido: <strong>{pct_dia_esperado:.1f}%</strong>
    </div>
    <div style="display: flex; align-items: center; margin-bottom: 8px;">
        <div style="width: 16px; height: 16px; background-color: {semaforo_colores['verde fuerte']}; border-radius: 50%; margin-right: 10px; box-shadow: 0 0 3px rgba(0,0,0,0.3);"></div>
        <span style="color: #e0e0e0; font-size: 11px;">Verde Fuerte: Avance >= {verde_fuerte_umbral:.1f}%</span>
    </div>
    <div style="display: flex; align-items: center; margin-bottom: 8px;">
        <div style="width: 16px; height: 16px; background-color: {semaforo_colores['verde claro']}; border-radius: 50%; margin-right: 10px; box-shadow: 0 0 3px rgba(0,0,0,0.3);"></div>
        <span style="color: #e0e0e0; font-size: 11px;">Verde Claro: Avance >= {verde_claro_umbral:.1f}%</span>
    </div>
    <div style="display: flex; align-items: center; margin-bottom: 8px;">
        <div style="width: 16px; height: 16px; background-color: {semaforo_colores['amarillo']}; border-radius: 50%; margin-right: 10px; box-shadow: 0 0 3px rgba(0,0,0,0.3);"></div>
        <span style="color: #e0e0e0; font-size: 11px;">Amarillo: Avance >= {amarillo_umbral:.1f}%</span>
    </div>
    <div style="display: flex; align-items: center;">
        <div style="width: 16px; height: 16px; background-color: {semaforo_colores['rojo']}; border-radius: 50%; margin-right: 10px; box-shadow: 0 0 3px rgba(0,0,0,0.3);"></div>
        <span style="color: #e0e0e0; font-size: 11px;">Rojo: Avance < {amarillo_umbral:.1f}%</span>
    </div>
</div>

<div class="footer">
    Datos al corte | Avance basado en meta esperada al dia de hoy
</div>

<script>
var unidadesData = {unidades_data_json};
var selectedClues = null;
var currentViewState = {{
    longitude: {lon_centro},
    latitude: {lat_centro},
    zoom: 5.5,
    pitch: 55,
    bearing: 0
}};

function updateLayers() {{
    var capaNeon = [];
    
    if (selectedClues) {{
        var unidadSeleccionada = unidadesData.find(function(d) {{ return d.clues === selectedClues; }});
        if (unidadSeleccionada) {{
            capaNeon = [
                new deck.ScatterplotLayer({{
                    id: "neon-glow",
                    data: [unidadSeleccionada],
                    getPosition: function(d) {{ return [d.lng, d.lat]; }},
                    radiusScale: 1,
                    radiusMinPixels: 40,
                    radiusMaxPixels: 80,
                    getRadius: 500,
                    getFillColor: [0, 255, 255, 120],
                    stroked: false,
                    pickable: false,
                    parameters: {{
                        blend: true,
                        blendFunc: [WebGLRenderingContext.SRC_ALPHA, WebGLRenderingContext.ONE],
                    }}
                }})
            ];
            
            var seleccionDiv = document.getElementById("seleccionInfo");
            seleccionDiv.innerHTML = "Seleccionado: " + unidadSeleccionada.clues + " - " + unidadSeleccionada.nombre.substring(0, 30);
            seleccionDiv.style.display = "block";
        }}
    }} else {{
        document.getElementById("seleccionInfo").style.display = "none";
    }}
    
    deckgl.setProps({{
        layers: [
            new deck.ColumnLayer({{
                id: "columnas-avance",
                data: unidadesData,
                diskResolution: 12,
                radius: 400,
                extruded: true,
                pickable: true,
                elevationScale: 0.5,
                getPosition: function(d) {{ return [d.lng, d.lat]; }},
                getElevation: function(d) {{ return d.altura; }},
                // IMPORTANTE: Aquí se define el color - AHORA USA COLOR FIJO
                getFillColor: function(d) {{
                    if (selectedClues && selectedClues === d.clues) {{
                        return [78, 205, 196, 255];  // Color cyan para selección
                    }}
                    // USAR COLORES FIJOS DEL SEMÁFORO (R,G,B ya vienen de Python)
                    return [d.r, d.g, d.b, 200];
                }}
            }}),
            ...capaNeon
        ]
    }});
}}

var deckgl = new deck.DeckGL({{
    container: "map",
    mapStyle: "https://basemaps.cartocdn.com/gl/dark-matter-gl-style/style.json",
    initialViewState: currentViewState,
    controller: true,
    getTooltip: function(info) {{
        if (!info.object) return null;
        var d = info.object;
        return {{
            html: '<div style="font-family: Segoe UI, Arial, sans-serif; min-width:260px; padding: 10px;">' +
                '<div style="font-weight:bold;font-size:13px;margin-bottom:6px;border-bottom: 1px solid rgba(78,205,196,0.3);padding-bottom: 4px;color:#4ecdc4;">' +
                d.nombre_de_la_unidad +
                '</div>' +
                '<div style="display: grid; grid-template-columns: 85px 1fr; gap: 4px; font-size: 11px;">' +
                '<span style="color:#aaa;">CLUES:</span>' +
                '<span style="font-weight:500;">' + d.clues + '</span>' +
                '<span style="color:#aaa;">Entidad:</span>' +
                '<span>' + d.entidad + '</span>' +
                '<span style="color:#aaa;">Nivel:</span>' +
                '<span>' + d.nivel + '</span>' +
                '<span style="color:#aaa;">Indicador:</span>' +
                '<span>' + d.tipo + '</span>' +
                '<span style="color:#aaa;">Realizado:</span>' +
                '<span style="font-weight:500;">' + d.realizado + '</span>' +
                '<span style="color:#aaa;">Meta Anual:</span>' +
                '<span>' + d.meta_anual + '</span>' +
                '<span style="color:#aaa;">Meta Esperada:</span>' +
                '<span>' + d.meta_esperada + '</span>' +
                '<span style="color:#aaa;">Avance Real:</span>' +
                '<span style="color:' + d.color_hex + '; font-weight:bold;">' + d.avance + '</span>' +
                '<span style="color:#aaa;">Avance Esperado:</span>' +
                '<span>' + d.pct_dia_fmt + '</span>' +
                '</div>' +
                '<div style="margin-top: 6px; padding-top: 4px; border-top: 1px solid rgba(78,205,196,0.2); font-size: 10px; color: #4ecdc4;">' +
                d.estado_semaforo +
                '</div></div>'
        }};
    }},
    layers: [
        new deck.ColumnLayer({{
            id: "columnas-avance",
            data: unidadesData,
            diskResolution: 12,
            radius: 400,
            extruded: true,
            pickable: true,
            elevationScale: 0.5,
            getPosition: function(d) {{ return [d.lng, d.lat]; }},
            getElevation: function(d) {{ return d.altura; }},
            // IMPORTANTE: Aquí también se define el color
            getFillColor: function(d) {{
                if (selectedClues && selectedClues === d.clues) {{
                    return [78, 205, 196, 255];
                }}
                return [d.r, d.g, d.b, 200];
            }}
        }})
    ]
}});

var buscadorInput = document.getElementById("buscadorClues");
var resultadosDiv = document.getElementById("resultadosBusqueda");

function buscarUnidades(texto) {{
    if (texto.length < 2) {{
        resultadosDiv.style.display = "none";
        return [];
    }}
    texto = texto.toLowerCase();
    return unidadesData.filter(function(u) {{
        return (
            u.clues.toLowerCase().includes(texto) ||
            u.nombre.toLowerCase().includes(texto)
        );
    }}).slice(0, 10);
}}

function centrarEnUnidad(unidad) {{
    selectedClues = unidad.clues;
    
    deckgl.setProps({{
        initialViewState: {{
            longitude: unidad.lng,
            latitude: unidad.lat,
            zoom: 14,
            pitch: 60,
            bearing: 0,
            transitionDuration: 1500,
            transitionInterpolator: new deck.FlyToInterpolator({{ speed: 1.2 }})
        }}
    }});
    
    updateLayers();
    
    buscadorInput.value = unidad.clues;
    resultadosDiv.style.display = "none";
}}

function mostrarResultados(resultados) {{
    resultadosDiv.innerHTML = "";
    if (resultados.length === 0) {{
        resultadosDiv.style.display = "none";
        return;
    }}
    resultados.forEach(function(u) {{
        var item = document.createElement("div");
        item.className = "resultado-item";
        item.innerHTML = 
            '<div style="font-weight:bold;color:#4ecdc4;font-size:13px;">' + u.clues + '</div>' +
            '<div style="font-size:11px;color:white;margin-top:3px;">' + u.nombre.substring(0,50) + '</div>' +
            '<div style="font-size:10px;color:#4ecdc4;margin-top:5px;">Avance: ' + u.avance + '</div>';
        item.onclick = function() {{ centrarEnUnidad(u); }};
        resultadosDiv.appendChild(item);
    }});
    resultadosDiv.style.display = "block";
}}

buscadorInput.addEventListener("input", function(e) {{
    var resultados = buscarUnidades(e.target.value);
    mostrarResultados(resultados);
}});

document.addEventListener("click", function(e) {{
    if (buscadorInput && resultadosDiv) {{
        if (!buscadorInput.contains(e.target) && !resultadosDiv.contains(e.target)) {{
            resultadosDiv.style.display = "none";
        }}
    }}
}});

deckgl.canvas.addEventListener('click', function(event) {{
    var pickInfo = deckgl.pickObject({{x: event.clientX, y: event.clientY}});
    if (pickInfo && pickInfo.object) {{
        if (selectedClues === pickInfo.object.clues) {{
            selectedClues = null;
        }} else {{
            selectedClues = pickInfo.object.clues;
        }}
        updateLayers();
    }}
}});
</script>
</body>
</html>
"""
# ============================================================
# GUARDAR Y ABRIR
# ============================================================

archivo_salida = Path("mapa_hospital_3d.html")

with open(archivo_salida, "w", encoding="utf-8") as f:
    f.write(html_3d)

webbrowser.open(archivo_salida.resolve().as_uri())

print("Mapa 3D creado exitosamente!")
print(f"Ubicacion: {archivo_salida.resolve()}")

Mapa 3D creado exitosamente!
Ubicacion: C:\Users\jose.valdez\OneDrive - IMSS-BIENESTAR\mapa_hospital_3d.html


## seccion de carreteras

In [ ]:
import geopandas as gpd

In [ ]:
carretera = gpd.read_file(fr"C:\Users\{usuario}\Downloads\794551163030_gpk\conjunto_de_datos\rnc2025.gpkg")  

c:\Users\jose.valdez\AppData\Local\Programs\Python\Python313\Lib\site-packages\pyogrio\geopandas.py:275: UserWarning: More than one layer found in 'rnc2025.gpkg': 'red_vial' (default), 'maniobra_prohibida', 'union_p', 'transbordador', 'estructura', 'puente', 'plaza_cobro', 'localidad', 'poste_de_referencia', 'sitio_de_interes', 'tarifas', 'tred_localidad', 'tred_sitio_de_interes'. Specify layer parameter to avoid this warning.
  result = read_func(


In [ ]:
conteo_carretera = carretera['TIPO_VIAL'].unique()    

In [ ]:
carretera = carretera[(carretera["TIPO_VIAL"] == "Carretera") & (carretera["CONDICION"] == "En operación") & (carretera["ESTATUS"] == "Habilitado")]

In [ ]:
len(carretera)

333806

In [ ]:
col = ['NOMBRE', 'TIPO_VIAL', 'LONGITUD', 'geometry']
carretera = carretera[col]

In [ ]:
carretera

,NOMBRE,TIPO_VIAL,LONGITUD,geometry
31,La Venta - Lechería,Carretera,185.741033,"LINESTRING (-99.27719 19.54568, -99.27717 19.5..."
58,La Venta - Lechería,Carretera,1321.556725,"LINESTRING (-99.28034 19.53304, -99.28048 19.5..."
93,Naucalpan - Toluca,Carretera,702.414829,"LINESTRING (-99.27931 19.44986, -99.27904 19.4..."
125,Naucalpan - Toluca,Carretera,187.621432,"LINESTRING (-99.25616 19.45917, -99.25583 19.4..."
127,Jiquipilco - Naucalpan,Carretera,145.907086,"LINESTRING (-99.39081 19.50262, -99.39078 19.5..."
...,...,...,...,...
4349682,San Hipólito - Xalapa,Carretera,99.938905,"LINESTRING (-97.66489 19.10703, -97.6649 19.10..."
4349686,Cuapiaxtla - Cuesta Blanca - Cuacnopalan,Carretera,106.040393,"LINESTRING (-97.78615 19.30034, -97.78604 19.2..."
4349687,Cuapiaxtla - Cuesta Blanca - Cuacnopalan,Carretera,79.570562,"LINESTRING (-97.78589 19.29941, -97.78584 19.2..."
4349688,Cuapiaxtla - Cuesta Blanca - Cuacnopalan,Carretera,26.661032,"LINESTRING (-97.6718 19.11024, -97.67168 19.11..."


## prueba de carreteras 

In [ ]:
# ============================================================
# PREPARACION DE DATOS PARA 3D (USANDO TU SEMÁFORO)
# ============================================================

df_3d = df_validos.copy()

# Calcular el porcentaje del año transcurrido (como ya lo haces)
if "pct_tiempo" in df_3d.columns and len(df_3d) > 0:
    pct_tiempo_decimal = pd.to_numeric(df_3d["pct_tiempo"].iloc[0], errors="coerce")
    if pd.isna(pct_tiempo_decimal) or np.isinf(pct_tiempo_decimal):
        pct_tiempo_decimal = 0
else:
    pct_tiempo_decimal = 0

pct_dia_esperado = pct_tiempo_decimal * 100  # Esto da 41.9% en tu ejemplo

# ¡IMPORTANTE! Actualizar la variable global para tus funciones
PCT_ANIO_PORCENTAJE = pct_dia_esperado

# Ahora sí, procesar los datos numéricos
if df_3d["pct_general"].dtype == "object":
    df_3d["pct_general_num"] = pd.to_numeric(df_3d["pct_general"].str.rstrip('%'), errors='coerce') / 100
else:
    if df_3d["pct_general"].max() > 1:
        df_3d["pct_general_num"] = df_3d["pct_general"] / 100
    else:
        df_3d["pct_general_num"] = df_3d["pct_general"]

df_3d["pct_general_num"] = df_3d["pct_general_num"].fillna(0).clip(0, 1)

# Altura de las columnas
df_3d["altura"] = 500 + (df_3d["pct_general_num"] * 20000)

# ========== USAR TU FUNCIÓN DE SEMÁFORO ==========
def hex_to_rgb(hex_color):
    hex_color = hex_color.lstrip('#')
    return tuple(int(hex_color[i:i+2], 16) for i in (0, 2, 4))

# Aplicar tu función obtener_color_semaforo (que ya usa PCT_ANIO_PORCENTAJE)
df_3d["color_semaforo_hex"] = df_3d["pct_general"].apply(obtener_color_semaforo)

# Convertir a RGB para deck.gl
df_3d[["r", "g", "b"]] = df_3d["color_semaforo_hex"].apply(
    lambda x: pd.Series(hex_to_rgb(x))
)

# Calcular los umbrales para mostrar en el HTML (los mismos que usa tu semáforo)
verde_fuerte_umbral = PCT_ANIO_PORCENTAJE * 0.95
verde_claro_umbral = PCT_ANIO_PORCENTAJE * 0.80
amarillo_umbral = PCT_ANIO_PORCENTAJE * 0.60

# ========== CORRECCIÓN: Calcular volúmenes con formato ==========
volumen_cirugias = pd.to_numeric(conteo_consultas.loc[0, "cirugia"], errors="coerce")
volumen_consultas = pd.to_numeric(conteo_consultas.loc[0, "consulta"], errors="coerce")

volumen_cirugias = 0 if pd.isna(volumen_cirugias) else volumen_cirugias
volumen_consultas = 0 if pd.isna(volumen_consultas) else volumen_consultas
volumen_realizado = volumen_cirugias + volumen_consultas

# Formatear para mostrar con separadores de miles
volumen_cirugias_fmt = f"{volumen_cirugias:,.0f}"
volumen_consultas_fmt = f"{volumen_consultas:,.0f}"
volumen_realizado_fmt = f"{volumen_realizado:,.0f}"

# Datos para el buscador (incluyendo el color correcto)
unidades_data = []
for _, row in df_3d.iterrows():
    unidades_data.append({
        "clues": row['clues_imb'],
        "nombre": row['nombre_de_la_unidad'],
        "lat": row['latitud'],
        "lng": row['longitud'],
        "avance": row['pct_general_fmt'],
        "entidad": row['entidad'],
        "nivel": row['nivel_atencion'],
        "tipo": row['tipo_indicador'],
        "realizado": row['realizado_fmt'],
        "meta_anual": row['meta_anual_fmt'],
        "meta_esperada": row['meta_esperada_fmt'],
        "pct_dia_fmt": row['pct_dia_fmt'],
        "color_hex": row['color_semaforo_hex'],
        "estado_semaforo": row['estado_semaforo'],
        "nombre_de_la_unidad": row['nombre_de_la_unidad'],
        "altura": row['altura'],
        "r": row['r'],
        "g": row['g'],
        "b": row['b']
    })
unidades_data_json = json.dumps(unidades_data, ensure_ascii=False)

# Calcular centro del mapa
lat_centro = df_3d["latitud"].mean()
lon_centro = df_3d["longitud"].mean()

# ============================================================
# GENERAR HTML 3D - VERSIÓN CON COLORES FIJOS DEL SEMÁFORO
# ============================================================

html_3d = f"""
<!DOCTYPE html>
<html>
<head>
<meta charset="utf-8"/>
<title>Hospital Dashboard 3D - Productividad por Unidad Medica</title>

<script src="https://unpkg.com/deck.gl@latest/dist.min.js"></script>
<script src="https://unpkg.com/maplibre-gl@latest/dist/maplibre-gl.js"></script>
<link href="https://unpkg.com/maplibre-gl@latest/dist/maplibre-gl.css" rel="stylesheet"/>

<style>
html, body {{
    margin: 0;
    width: 100%;
    height: 100%;
    font-family: 'Segoe UI', Arial, sans-serif;
    background: #0a0a0a;
}}

#map {{
    width: 100%;
    height: 100%;
    background: #0d0d0d;
}}

/* CABECERA MODIFICADA CON LOGO */
.header {{
    position: fixed;
    top: 0;
    left: 0;
    right: 0;
    z-index: 10000;
    background: linear-gradient(135deg, #0a0e27, #0f142e);
    color: white;
    padding: 8px 25px;
    font-size: 18px;
    font-weight: bold;
    text-align: center;
    box-shadow: 0 2px 10px rgba(0,0,0,0.5);
    pointer-events: none;
    border-bottom: 1px solid rgba(78,205,196,0.3);
    display: flex;
    align-items: center;
    justify-content: center;
    gap: 15px;
}}

.header-logo {{
    height: 35px;
    width: auto;
    filter: brightness(0) invert(1);
}}

.header-text {{
    letter-spacing: 1px;
}}

.kpi-container {{
    position: fixed;
    top: 65px;
    left: 0;
    right: 0;
    z-index: 10000;
    display: grid;
    grid-template-columns: repeat(4, 1fr);
    gap: 12px;
    padding: 12px 20px;
    pointer-events: none;
}}

.kpi-card {{
    background: rgba(10,10,10,0.85);
    backdrop-filter: blur(10px);
    padding: 10px 15px;
    border-radius: 10px;
    border: 1px solid rgba(78,205,196,0.3);
    pointer-events: auto;
    box-shadow: 0 2px 5px rgba(0,0,0,0.3);
}}

.kpi-label {{
    font-size: 10px;
    color: #4ecdc4;
    text-transform: uppercase;
    letter-spacing: 1px;
}}

.kpi-value {{
    font-size: 24px;
    font-weight: bold;
    color: white;
}}

.kpi-sub {{
    font-size: 9px;
    color: #888;
}}

.dashboard-btn {{
    position: fixed;
    top: 350px;
    left: 20px;
    z-index: 10001;
}}

.dashboard-btn button {{
    background: linear-gradient(135deg, #4ecdc4, #2c3e50);
    color: white;
    border: none;
    padding: 8px 18px;
    border-radius: 25px;
    font-size: 12px;
    font-weight: bold;
    cursor: pointer;
    font-family: 'Segoe UI', Arial, sans-serif;
    box-shadow: 0 2px 5px rgba(0,0,0,0.3);
    transition: all 0.3s;
}}

.dashboard-btn button:hover {{
    transform: translateY(-2px);
    box-shadow: 0 4px 10px rgba(78,205,196,0.3);
}}

.buscador {{
    position: fixed;
    top: 200px;
    left: 20px;
    z-index: 10001;
    width: 320px;
}}

.buscador-box {{
    background: rgba(10,10,10,0.85);
    backdrop-filter: blur(10px);
    border-radius: 12px;
    padding: 12px;
    border: 1px solid rgba(78,205,196,0.3);
    box-shadow: 0 4px 15px rgba(0,0,0,0.5);
}}

.buscador-label {{
    color: #4ecdc4;
    font-size: 10px;
    text-transform: uppercase;
    margin-bottom: 8px;
    letter-spacing: 1px;
}}

.buscador-input {{
    width: 100%;
    padding: 10px 15px;
    background: rgba(20,20,20,0.9);
    border: 1px solid rgba(78,205,196,0.3);
    border-radius: 8px;
    color: white;
    font-size: 13px;
    outline: none;
    font-family: 'Segoe UI', Arial, sans-serif;
    box-sizing: border-box;
}}

.buscador-input:focus {{
    border-color: #4ecdc4;
    box-shadow: 0 0 5px rgba(78,205,196,0.5);
}}

.resultados {{
    background: rgba(10,10,10,0.95);
    border-radius: 12px;
    margin-top: 8px;
    max-height: 350px;
    overflow-y: auto;
    display: none;
    border: 1px solid rgba(78,205,196,0.3);
    box-shadow: 0 4px 15px rgba(0,0,0,0.5);
}}

.resultado-item {{
    padding: 12px 15px;
    cursor: pointer;
    border-bottom: 1px solid rgba(78,205,196,0.1);
    font-family: 'Segoe UI', Arial, sans-serif;
    transition: all 0.2s;
}}

.resultado-item:hover {{
    background: rgba(78,205,196,0.15);
    border-left: 3px solid #4ecdc4;
}}

.semaforo {{
    position: fixed;
    bottom: 20px;
    right: 20px;
    z-index: 10001;
    background: rgba(10,10,10,0.85);
    backdrop-filter: blur(10px);
    padding: 12px 18px;
    border-radius: 12px;
    min-width: 240px;
    border: 1px solid rgba(78,205,196,0.3);
    box-shadow: 0 4px 15px rgba(0,0,0,0.5);
    pointer-events: auto;
}}

.footer {{
    position: fixed;
    bottom: 20px;
    left: 20px;
    z-index: 10001;
    background: rgba(10,10,10,0.8);
    backdrop-filter: blur(10px);
    padding: 8px 12px;
    border-radius: 8px;
    font-size: 10px;
    color: #888;
    border: 1px solid rgba(78,205,196,0.2);
    pointer-events: auto;
}}

.deck-tooltip {{
    background: rgba(0,0,0,0.85) !important;
    backdrop-filter: blur(8px) !important;
    border: 1px solid rgba(78,205,196,0.4) !important;
    border-radius: 8px !important;
    box-shadow: 0 4px 15px rgba(0,0,0,0.5) !important;
    padding: 0 !important;
}}

.seleccion-activa {{
    position: fixed;
    top: 420px;
    left: 20px;
    z-index: 10001;
    background: rgba(78,205,196,0.2);
    backdrop-filter: blur(10px);
    padding: 5px 12px;
    border-radius: 20px;
    border: 1px solid #4ecdc4;
    color: #4ecdc4;
    font-size: 11px;
    font-weight: bold;
    pointer-events: none;
}}
</style>
</head>
<body>

<div id="map"></div>

<!-- CABECERA CON LOGO DEL IMSS BIENESTAR -->
<div class="header">
    <img class="header-logo" 
         src="https://imssbienestar.gob.mx/assets/img/imb_b.svg" 
         alt="Logo IMSS-BIENESTAR"
         onerror="this.onerror=null; this.style.display='none';">
    <div class="header-text">Hospital Dashboard - Productividad por Unidad Medica (3D)</div>
</div>

<div class="kpi-container">
    <div class="kpi-card">
        <div class="kpi-label">Unidades Activas</div>
        <div class="kpi-value">{total_unidades_activas:,}</div>
        <div class="kpi-sub">Cirugia: {unidades_cirugias} | Consulta: {unidades_consultas}</div>
    </div>
    <div class="kpi-card">
        <div class="kpi-label">Avance General</div>
        <div class="kpi-value">{avg_avance_general:.1f}%</div>
        <div class="kpi-sub">Cirugia: {avg_avance_cirugias:.1f}% | Consulta: {avg_avance_consultas:.1f}%</div>
    </div>
    <div class="kpi-card">
        <div class="kpi-label">Volumen Realizado</div>
        <div class="kpi-value">{volumen_realizado_fmt}</div>
        <div class="kpi-sub">Cirugias: {volumen_cirugias_fmt} | Consultas: {volumen_consultas_fmt}</div>
    </div>
    <div class="kpi-card">
        <div class="kpi-label">Cobertura</div>
        <div class="kpi-value">{df_validos['entidad'].nunique()}</div>
        <div class="kpi-sub">Entidades Federativas</div>
    </div>
</div>

<div class="dashboard-btn">
    <button onclick="window.open('https://argontc.shinyapps.io/pptx/', '_blank')">
        Ver Dashboard de Reportes
    </button>
</div>

<div class="buscador">
    <div class="buscador-box">
        <div class="buscador-label">Buscador de Unidades</div>
        <input type="text" id="buscadorClues" class="buscador-input" placeholder="Buscar por CLUES o nombre...">
    </div>
    <div id="resultadosBusqueda" class="resultados"></div>
</div>

<div id="seleccionInfo" class="seleccion-activa" style="display: none;"></div>

<div class="semaforo">
    <div style="color: white; font-size: 13px; font-weight: bold; margin-bottom: 10px; text-align: center; border-bottom: 1px solid rgba(78,205,196,0.3); padding-bottom: 5px;">
        Semaforo de Avance
    </div>
    <div style="font-size: 11px; color: #4ecdc4; margin-bottom: 10px; text-align: center; background: rgba(0,0,0,0.5); padding: 5px; border-radius: 6px;">
        Ano transcurrido: <strong>{pct_dia_esperado:.1f}%</strong>
    </div>
    <div style="display: flex; align-items: center; margin-bottom: 8px;">
        <div style="width: 16px; height: 16px; background-color: {semaforo_colores['verde fuerte']}; border-radius: 50%; margin-right: 10px; box-shadow: 0 0 3px rgba(0,0,0,0.3);"></div>
        <span style="color: #e0e0e0; font-size: 11px;">Verde Fuerte: Avance >= {verde_fuerte_umbral:.1f}%</span>
    </div>
    <div style="display: flex; align-items: center; margin-bottom: 8px;">
        <div style="width: 16px; height: 16px; background-color: {semaforo_colores['verde claro']}; border-radius: 50%; margin-right: 10px; box-shadow: 0 0 3px rgba(0,0,0,0.3);"></div>
        <span style="color: #e0e0e0; font-size: 11px;">Verde Claro: Avance >= {verde_claro_umbral:.1f}%</span>
    </div>
    <div style="display: flex; align-items: center; margin-bottom: 8px;">
        <div style="width: 16px; height: 16px; background-color: {semaforo_colores['amarillo']}; border-radius: 50%; margin-right: 10px; box-shadow: 0 0 3px rgba(0,0,0,0.3);"></div>
        <span style="color: #e0e0e0; font-size: 11px;">Amarillo: Avance >= {amarillo_umbral:.1f}%</span>
    </div>
    <div style="display: flex; align-items: center;">
        <div style="width: 16px; height: 16px; background-color: {semaforo_colores['rojo']}; border-radius: 50%; margin-right: 10px; box-shadow: 0 0 3px rgba(0,0,0,0.3);"></div>
        <span style="color: #e0e0e0; font-size: 11px;">Rojo: Avance < {amarillo_umbral:.1f}%</span>
    </div>
</div>

<div class="footer">
    Datos al corte | Avance basado en meta esperada al dia de hoy
</div>

<script>
var unidadesData = {unidades_data_json};
var selectedClues = null;
var currentViewState = {{
    longitude: {lon_centro},
    latitude: {lat_centro},
    zoom: 5.5,
    pitch: 55,
    bearing: 0
}};

function updateLayers() {{
    var capaNeon = [];
    
    if (selectedClues) {{
        var unidadSeleccionada = unidadesData.find(function(d) {{ return d.clues === selectedClues; }});
        if (unidadSeleccionada) {{
            capaNeon = [
                new deck.ScatterplotLayer({{
                    id: "neon-glow",
                    data: [unidadSeleccionada],
                    getPosition: function(d) {{ return [d.lng, d.lat]; }},
                    radiusScale: 1,
                    radiusMinPixels: 40,
                    radiusMaxPixels: 80,
                    getRadius: 500,
                    getFillColor: [0, 255, 255, 120],
                    stroked: false,
                    pickable: false,
                    parameters: {{
                        blend: true,
                        blendFunc: [WebGLRenderingContext.SRC_ALPHA, WebGLRenderingContext.ONE],
                    }}
                }})
            ];
            
            var seleccionDiv = document.getElementById("seleccionInfo");
            seleccionDiv.innerHTML = "Seleccionado: " + unidadSeleccionada.clues + " - " + unidadSeleccionada.nombre.substring(0, 30);
            seleccionDiv.style.display = "block";
        }}
    }} else {{
        document.getElementById("seleccionInfo").style.display = "none";
    }}
    
    deckgl.setProps({{
        layers: [
            new deck.ColumnLayer({{
                id: "columnas-avance",
                data: unidadesData,
                diskResolution: 12,
                radius: 400,
                extruded: true,
                pickable: true,
                elevationScale: 0.5,
                getPosition: function(d) {{ return [d.lng, d.lat]; }},
                getElevation: function(d) {{ return d.altura; }},
                getFillColor: function(d) {{
                    if (selectedClues && selectedClues === d.clues) {{
                        return [78, 205, 196, 255];
                    }}
                    return [d.r, d.g, d.b, 200];
                }}
            }}),
            ...capaNeon
        ]
    }});
}}

var deckgl = new deck.DeckGL({{
    container: "map",
    mapStyle: "https://basemaps.cartocdn.com/gl/dark-matter-gl-style/style.json",
    initialViewState: currentViewState,
    controller: true,
    getTooltip: function(info) {{
        if (!info.object) return null;
        var d = info.object;
        return {{
            html: '<div style="font-family: Segoe UI, Arial, sans-serif; min-width:260px; padding: 10px;">' +
                '<div style="font-weight:bold;font-size:13px;margin-bottom:6px;border-bottom: 1px solid rgba(78,205,196,0.3);padding-bottom: 4px;color:#4ecdc4;">' +
                d.nombre_de_la_unidad +
                '</div>' +
                '<div style="display: grid; grid-template-columns: 85px 1fr; gap: 4px; font-size: 11px;">' +
                '<span style="color:#aaa;">CLUES:</span>' +
                '<span style="font-weight:500;">' + d.clues + '</span>' +
                '<span style="color:#aaa;">Entidad:</span>' +
                '<span>' + d.entidad + '</span>' +
                '<span style="color:#aaa;">Nivel:</span>' +
                '<span>' + d.nivel + '</span>' +
                '<span style="color:#aaa;">Indicador:</span>' +
                '<span>' + d.tipo + '</span>' +
                '<span style="color:#aaa;">Realizado:</span>' +
                '<span style="font-weight:500;">' + d.realizado + '</span>' +
                '<span style="color:#aaa;">Meta Anual:</span>' +
                '<span>' + d.meta_anual + '</span>' +
                '<span style="color:#aaa;">Meta Esperada:</span>' +
                '<span>' + d.meta_esperada + '</span>' +
                '<span style="color:#aaa;">Avance Real:</span>' +
                '<span style="color:' + d.color_hex + '; font-weight:bold;">' + d.avance + '</span>' +
                '<span style="color:#aaa;">Avance Esperado:</span>' +
                '<span>' + d.pct_dia_fmt + '</span>' +
                '</div>' +
                '<div style="margin-top: 6px; padding-top: 4px; border-top: 1px solid rgba(78,205,196,0.2); font-size: 10px; color: #4ecdc4;">' +
                d.estado_semaforo +
                '</div></div>'
        }};
    }},
    layers: [
        new deck.ColumnLayer({{
            id: "columnas-avance",
            data: unidadesData,
            diskResolution: 12,
            radius: 400,
            extruded: true,
            pickable: true,
            elevationScale: 0.5,
            getPosition: function(d) {{ return [d.lng, d.lat]; }},
            getElevation: function(d) {{ return d.altura; }},
            getFillColor: function(d) {{
                if (selectedClues && selectedClues === d.clues) {{
                    return [78, 205, 196, 255];
                }}
                return [d.r, d.g, d.b, 200];
            }}
        }})
    ]
}});

var buscadorInput = document.getElementById("buscadorClues");
var resultadosDiv = document.getElementById("resultadosBusqueda");

function buscarUnidades(texto) {{
    if (texto.length < 2) {{
        resultadosDiv.style.display = "none";
        return [];
    }}
    texto = texto.toLowerCase();
    return unidadesData.filter(function(u) {{
        return (
            u.clues.toLowerCase().includes(texto) ||
            u.nombre.toLowerCase().includes(texto)
        );
    }}).slice(0, 10);
}}

function centrarEnUnidad(unidad) {{
    selectedClues = unidad.clues;
    
    deckgl.setProps({{
        initialViewState: {{
            longitude: unidad.lng,
            latitude: unidad.lat,
            zoom: 14,
            pitch: 60,
            bearing: 0,
            transitionDuration: 1500,
            transitionInterpolator: new deck.FlyToInterpolator({{ speed: 1.2 }})
        }}
    }});
    
    updateLayers();
    
    buscadorInput.value = unidad.clues;
    resultadosDiv.style.display = "none";
}}

function mostrarResultados(resultados) {{
    resultadosDiv.innerHTML = "";
    if (resultados.length === 0) {{
        resultadosDiv.style.display = "none";
        return;
    }}
    resultados.forEach(function(u) {{
        var item = document.createElement("div");
        item.className = "resultado-item";
        item.innerHTML = 
            '<div style="font-weight:bold;color:#4ecdc4;font-size:13px;">' + u.clues + '</div>' +
            '<div style="font-size:11px;color:white;margin-top:3px;">' + u.nombre.substring(0,50) + '</div>' +
            '<div style="font-size:10px;color:#4ecdc4;margin-top:5px;">Avance: ' + u.avance + '</div>';
        item.onclick = function() {{ centrarEnUnidad(u); }};
        resultadosDiv.appendChild(item);
    }});
    resultadosDiv.style.display = "block";
}}

buscadorInput.addEventListener("input", function(e) {{
    var resultados = buscarUnidades(e.target.value);
    mostrarResultados(resultados);
}});

document.addEventListener("click", function(e) {{
    if (buscadorInput && resultadosDiv) {{
        if (!buscadorInput.contains(e.target) && !resultadosDiv.contains(e.target)) {{
            resultadosDiv.style.display = "none";
        }}
    }}
}});

deckgl.canvas.addEventListener('click', function(event) {{
    var pickInfo = deckgl.pickObject({{x: event.clientX, y: event.clientY}});
    if (pickInfo && pickInfo.object) {{
        if (selectedClues === pickInfo.object.clues) {{
            selectedClues = null;
        }} else {{
            selectedClues = pickInfo.object.clues;
        }}
        updateLayers();
    }}
}});
</script>
</body>
</html>
"""

# ============================================================
# GUARDAR Y ABRIR
# ============================================================

archivo_salida = Path("index.html")

with open(archivo_salida, "w", encoding="utf-8") as f:
    f.write(html_3d)

webbrowser.open(archivo_salida.resolve().as_uri())

print("Mapa 3D creado exitosamente!")
print(f"Ubicacion: {archivo_salida.resolve()}")

Mapa 3D creado exitosamente!
Ubicacion: C:\Users\jose.valdez\OneDrive - IMSS-BIENESTAR\index.html


In [ ]:
import os
print(f"Directorio actual: {os.getcwd()}")
print(f"Ruta completa del archivo: {Path('index.html').resolve()}")

Directorio actual: c:\Users\jose.valdez\OneDrive - IMSS-BIENESTAR
Ruta completa del archivo: C:\Users\jose.valdez\OneDrive - IMSS-BIENESTAR\index.html
